# M2 (DeepAR) e M3 (TimeGrad) su GPU Colab

Questo notebook fa girare i due modelli **pesanti** della ladder — **M2 DeepAR** e **M3 TimeGrad** —
su **GPU Colab**, sui dataset **Exchange** (piccolo, per fare pratica) e **Electricity** (quello vero, D=321).

I run leggeri (M0 seasonal-naive, M1 ARIMA) restano sul laptop: sono CPU-bound e su Colab non
andrebbero piu` veloci. Qui mettiamo solo il carico che la GPU accelera davvero.

---

### Prima volta su Colab? Leggi qui
- Un notebook Colab e` come Jupyter, ma gira su una macchina di Google: **il tuo laptop resta libero**.
- Esegui una cella con **Shift+Invio**, oppure tutto in fila da **Runtime > Run all**.
- La macchina e` **effimera**: quando chiudi la sessione sparisce tutto (codice, dati, risultati).
  Per questo allo **step 8** scarichiamo i risultati.
- Per la **GPU**: **Runtime > Change runtime type > Hardware accelerator > GPU (T4) > Save**.

### Ordine consigliato
1. Esegui tutto **prima con `DATASET = "exchange"`** (giro completo in pochi minuti):
   impari il flusso e **validiamo l'ambiente** (le versioni delle librerie).
2. Quando fila liscio, torna alla **cella 5**, metti `DATASET = "electricity"` e rilancia i run.


## 1 - Verifica la GPU
Se sotto vedi una tabella con una **Tesla T4** (o simile), la GPU e` attiva.
Se da` errore o non vedi GPU: **Runtime > Change runtime type > GPU**, poi riesegui.


In [ ]:
!nvidia-smi

## 2 - Scarica il codice del progetto
Cloniamo il repo del gruppo e ci mettiamo sul branch di lavoro.

> Il branch `feature/port-ladder-exchange` deve essere **su origin** perche' Colab possa clonarlo.
> Se la cella da` `Remote branch not found`, il branch non e` ancora stato pushato: avvisa e lo sistemiamo.


In [ ]:
REPO_URL = "https://github.com/Icaica14/pml-diffusion-tsf.git"
BRANCH   = "feature/port-ladder-exchange"

import os
if not os.path.isdir("/content/pml-diffusion-tsf"):
    !git clone --branch $BRANCH $REPO_URL /content/pml-diffusion-tsf
%cd /content/pml-diffusion-tsf
!git log --oneline -1

## 3 - Installa le librerie pesanti (versioni bloccate)
M2/M3 girano su **GluonTS 0.13** + **PyTorchTS** (commit compatibile) + il **PyTorch CUDA**
gia` presente su Colab (non lo tocchiamo, cosi` la GPU resta a posto).

**Attenzione:** questa cella fa il **downgrade di NumPy a < 2** e di **pandas a < 2.2**
(servono entrambi a GluonTS 0.13: con pandas >= 2.2 le sue feature di calendario rifiutano
la frequenza oraria di Electricity, `freq="h"`, con `Exception: invalid frequency`).
Quando finisce, Colab ti chiedera` (o farai tu) **Runtime > Restart session**: e` normale,
NumPy/pandas vanno ricaricati. **Dopo il restart NON rifare questa cella**: riparti dalla cella 4.

> Queste versioni sono un *candidato*: questo primo giro su Exchange serve **anche** a confermarle.
> Se qualcosa non installa, copiami l'errore e le aggiusto.

In [ ]:
# GluonTS 0.13 (+ lightning compatibile via extra [torch]), numpy<2 e pandas<2.2.
# NON reinstalliamo torch: usiamo quello CUDA gia' presente su Colab.
!pip install -q "gluonts[torch]==0.13.7" "numpy<2" "pandas<2.2"

# PyTorchTS (il backend di TimeGrad) al commit compatibile con gluonts 0.13.
# --no-deps cosi' non si ritira dietro una gluonts vecchia.
!pip install -q --no-deps "git+https://github.com/zalandoresearch/pytorch-ts.git@81be06bcc"

print("\nInstallazione finita. Ora: Runtime > Restart session, poi continua dalla cella 4.")

## 4 - Controllo ambiente (dopo il restart)
Importa tutto e stampa le versioni. Se arriva in fondo senza errori e vedi
`cuda disponibile: True`, sei pronto. Se esplode, copiami l'output.


In [ ]:
%cd /content/pml-diffusion-tsf
import numpy, pandas, torch, gluonts, pts
print("numpy   :", numpy.__version__)
print("pandas  :", pandas.__version__)
print("torch   :", torch.__version__)
print("gluonts :", gluonts.__version__)
print("cuda disponibile:", torch.cuda.is_available())
print("device  :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 5 - Scegli il dataset
Parti da `"exchange"` per fare pratica (giro completo in pochi minuti).
Quando tutto fila, metti `"electricity"` e **riesegui dalla cella 5 in giu`**.

`CHUNK` serve solo a Electricity (D=321): il tensore completo dei campioni sarebbe centinaia di GB,
quindi lo valutiamo a blocchi. Su Exchange (D=8) non serve, quindi 0 (eager).

> I dati si scaricano da soli al primo uso: `build_dataset` li prende dalla rete in `data/raw/`.


In [ ]:
DATASET = "exchange"          # poi cambia in: "electricity"
CONFIG  = f"configs/data_{DATASET}.yaml"
CHUNK   = 256 if DATASET == "electricity" else 0
# Budget di training. Su Electricity il fit costa poco e il collo di bottiglia e' il
# predict (campionamento), quindi alziamo le epoche "gratis" (convergenza + calibrazione);
# samples/diff_steps restano fermi per non gonfiare il predict. Exchange resta al "smoke" (20).
EPOCHS  = 50 if DATASET == "electricity" else 20
print(f"dataset={DATASET}  config={CONFIG}  chunk={CHUNK}  epochs={EPOCHS}")

## 6 - M2: DeepAR
Alleno un DeepAR globale e valuto sul **test** split. Su Exchange ~2-3 min; su Electricity di piu`,
ma sempre su GPU. La riga finisce nella registry del clone Colab: la riportiamo a casa allo step 8.

> Variante "tuned" (Exchange): aggiungi `--no-time-features` per togliere le calendar features
> (su Exchange la data e` nominale). Per fare pratica lascia pure il default.


In [ ]:
!python -m experiments.run_deepar --config $CONFIG --chunk $CHUNK --epochs $EPOCHS --accelerator gpu

## 7 - M3: TimeGrad (diffusion)
Il modello centrale del progetto: diffusione condizionata + RNN multivariato. E` il piu` delicato.
Se la prima volta da` un errore di shape o su `input_size`, copiamelo. Su GPU il pezzo lento e` il
campionamento dei 100 trajectory per finestra.

> **Electricity:** run lungo (predict ~2-3h, **niente resume** sul free tier). Fallo in una
> **sessione fresca** *dopo* aver gia` portato a casa M2 (step 8) — non di fila nello stesso run,
> o una disconnessione ti fa ributtare via ore.


In [ ]:
!python -m experiments.run_timegrad --config $CONFIG --chunk $CHUNK --epochs $EPOCHS --device cuda

## 8 - Porta a casa i risultati
Colab e` effimero. Salviamo **solo le righe nuove** (DeepAR / TimeGrad), cosi` non sovrascrivi
M0/M1 che hai in locale. Le stampiamo (per copia-incolla) e te le scarico come file.


In [ ]:
import pathlib
lines = pathlib.Path("results/registry.csv").read_text().splitlines()
header = lines[0]
NEW_MODELS = ("deepar", "deepar_notf", "timegrad")
new = [l for l in lines[1:] if len(l.split(",")) > 2 and l.split(",")[2] in NEW_MODELS]
out = "\n".join([header] + new) + "\n"
pathlib.Path("results/colab_new_rows.csv").write_text(out)
print(out)
print(f"--> {len(new)} righe nuove in results/colab_new_rows.csv")

In [ ]:
from google.colab import files
files.download("results/colab_new_rows.csv")

## 9 - E adesso?
- **Copiami l'output della cella 8** (o caricami `colab_new_rows.csv`): unisco quelle righe alla
  `results/registry.csv` del repo e committo io in locale.
- Per il run vero su **Electricity**: torna alla **cella 5**, metti `DATASET = "electricity"`,
  e riesegui **5 > 8**.
- Sui run lunghi tieni il **tab aperto**: sul free tier Colab disconnette dopo ~90 min di inattivita`.
